# Artefact integrity and supply-chain audit

**Module 15 — Supply Chain and Model Artefacts**

A model is a binary artefact with a name, a version and a hash. This notebook builds a small bundle, signs off
a manifest, tampers with one file to show the check working, and then replaces an unsafe pickle with
safetensors so the loader cannot execute code at all.

## Learning objectives

1. Compute and verify SHA-256 hashes over a multi-file artefact bundle.
2. Detect a single modified byte and locate the affected file from the manifest.
3. Explain why pickled weights are a code-execution risk and how safetensors removes it.
4. Record provenance fields that a reviewer can use without trusting the uploader.
5. Produce a bundle audit table suitable for a release gate.

## Background

Artefact integrity answers "is this the file we approved", not "is this file safe". Those are different
questions and need different controls:

* **Integrity** — hashes, signatures, immutable storage.
* **Safety** — format restrictions, sandboxed loading, dependency pinning.
* **Provenance** — who trained it, on what data, with which code and hyperparameters.

`torch.save`/`pickle` can execute arbitrary code during `load`, which is why loading an untrusted checkpoint is
equivalent to running an untrusted program. `safetensors` stores only tensors and metadata, so it cannot do
that.

## Dependencies and installation

```bash
python -m pip install "torch>=2.4" --index-url https://download.pytorch.org/whl/cpu
python -m pip install "safetensors>=0.4" "pandas>=2.2"
```

CPU, seconds. Files are written to a temporary directory and removed afterwards; nothing is downloaded.

In [1]:
import hashlib
import json
import tempfile
from pathlib import Path

import pandas as pd
import torch
from safetensors.torch import load_file, save_file

pd.set_option("display.width", 140)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — Build a bundle: weights, config, card

Three files, one manifest. The manifest is the thing you sign, not the individual files.

In [2]:
workdir = Path(tempfile.mkdtemp(prefix="sas-m15-"))
tensors = {"encoder.weight": torch.arange(12, dtype=torch.float32).reshape(3, 4),
           "head.bias": torch.zeros(2)}
save_file(tensors, str(workdir / "model.safetensors"))
(workdir / "config.json").write_text(json.dumps({"hidden": 4, "classes": 2}, indent=2), encoding="utf-8")
(workdir / "model-card.md").write_text("# Toy classifier\nTrained on synthetic data. Not for clinical use.\n",
                                       encoding="utf-8")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


manifest = {
    "bundle": "toy-classifier",
    "version": "1.0.0",
    "files": {p.name: {"sha256": sha256(p), "bytes": p.stat().st_size} for p in sorted(workdir.iterdir())},
}
print(json.dumps(manifest, indent=2)[:500])

{
  "bundle": "toy-classifier",
  "version": "1.0.0",
  "files": {
    "config.json": {
      "sha256": "20a06c96d64b9a153175e2d00164fdd1882aab483e73f7b105989fa9a8bb907e",
      "bytes": 36
    },
    "model-card.md": {
      "sha256": "0f1adbe0fbc1176e81a0e8e8c0358de1d437e853cc3a600a42200bc30cb7efeb",
      "bytes": 68
    },
    "model.safetensors": {
      "sha256": "8b140e0e50622a90f5ad49a4f7dcc494a94a3ef4950f6b48dfc663eac8c45cd0",
      "bytes": 200
    }
  }
}


**Expected output** — a manifest with a hash and byte count per file. Hashes are of content, so renaming a file
does not hide a change and changing one byte does not survive.

### Step 2 — Verify, then tamper

Verification is a loop over the manifest. One modified byte in `config.json` must fail, and the manifest must
name the file.

In [3]:
def verify(root, manifest):
    rows = []
    for name, expected in manifest["files"].items():
        path = Path(root) / name
        if not path.exists():
            rows.append({"file": name, "status": "missing"})
            continue
        actual = sha256(path)
        rows.append({"file": name, "status": "ok" if actual == expected["sha256"] else "MISMATCH",
                     "expected": expected["sha256"][:12], "actual": actual[:12]})
    return pd.DataFrame(rows)


clean = verify(workdir, manifest)
print(f"clean bundle: {int((clean.status == 'ok').sum())}/{len(clean)} ok")

config_path = workdir / "config.json"
config_path.write_text(json.dumps({"hidden": 8, "classes": 2}, indent=2), encoding="utf-8")
tampered = verify(workdir, manifest)
tampered

clean bundle: 3/3 ok


,file,status,expected,actual
0,config.json,MISMATCH,20a06c96d64b,112719a3f602
1,model-card.md,ok,0f1adbe0fbc1,0f1adbe0fbc1
2,model.safetensors,ok,8b140e0e5062,8b140e0e5062


**Expected output** — the clean check reports all files ok; after the edit, `config.json` reports `MISMATCH` with
different short hashes and the other files stay ok. The table points at the file, which is the operational
value: "the bundle is bad" is not actionable, "config.json changed" is.

In [4]:
config_path.write_text(json.dumps({"hidden": 4, "classes": 2}, indent=2), encoding="utf-8")
restored = verify(workdir, manifest)
assert (restored.status == "ok").all()
print("restored bundle verified:", int((restored.status == "ok").sum()), "files")

restored bundle verified: 3 files


**Expected output** — `restored bundle verified: 3 files`. Restoring the original bytes restores the hash, which
is the property that lets you re-download and re-verify.

### Step 3 — Why the format matters: pickle versus safetensors

A pickle can carry a `__reduce__` that runs a command on load. The demonstration here writes a harmless pickle
and shows *where the code would execute*, then shows safetensors refusing to represent such an object at all.

In [5]:
import pickle


class Loaded:
    def __init__(self, message):
        self.message = message

    def __reduce__(self):
        # In real attacks this tuple runs a shell command. Here it only builds an object.
        return (Loaded, ("this ran during unpickling",))


pickle_path = workdir / "unsafe.pkl"
with open(pickle_path, "wb") as fh:
    pickle.dump(Loaded("payload"), fh)

with open(pickle_path, "rb") as fh:
    obj = pickle.load(fh)          # constructor side effect happens here
print("pickle load returned:", obj.message)

try:
    save_file({"bad": Loaded("x")}, str(workdir / "bad.safetensors"))
    print("safetensors accepted a non-tensor object (unexpected)")
except Exception as exc:
    print("safetensors refused:", type(exc).__name__)

pickle load returned: this ran during unpickling
safetensors refused: ValueError


**Expected output** — `pickle load returned: this ran during unpickling` and `safetensors refused: ...`. The
pickle executed a constructor during load; the format is the vulnerability, not the code that called it. The
safetensors refusal is the property you want: the format cannot represent executable objects.

### Step 4 — Provenance record a reviewer can act on

Integrity says the bytes are right. Provenance says those bytes should exist. Fill the fields once, at build
time; they are worthless if reconstructed afterwards.

In [6]:
provenance = pd.DataFrame(
    [
        {"field": "model.digest", "value": manifest["files"]["model.safetensors"]["sha256"][:16], "why": "identifies the artefact"},
        {"field": "config.digest", "value": manifest["files"]["config.json"]["sha256"][:16], "why": "pins architecture"},
        {"field": "training.data_digest", "value": "sha256:9f2c...synthetic-v3", "why": "ties weights to a dataset version"},
        {"field": "code.commit", "value": "git:4ac1e2f", "why": "reproduces the build"},
        {"field": "base.image", "value": "python:3.12-slim@sha256:...", "why": "pins the runtime"},
        {"field": "format", "value": "safetensors", "why": "excludes executable pickle payloads"},
    ]
)
provenance

,field,value,why
0,model.digest,8b140e0e50622a90,identifies the artefact
1,config.digest,20a06c96d64b9a15,pins architecture
2,training.data_digest,sha256:9f2c...synthetic-v3,ties weights to a dataset version
3,code.commit,git:4ac1e2f,reproduces the build
4,base.image,python:3.12-slim@sha256:...,pins the runtime
5,format,safetensors,excludes executable pickle payloads


**Expected output** — six fields. Notice that three of them are digests: model, data, base image. A release gate
that checks only the model digest verifies one third of the supply chain.

### Step 5 — The audit gate

One function that answers "may this bundle be promoted", returning per-check status. This is what the gate in
Module 20 calls.

In [7]:
def audit(root, manifest, require_format="safetensors"):
    checks = []
    table = verify(root, manifest)
    checks.append({"check": "all file hashes match", "pass": bool((table.status == "ok").all())})
    checks.append({"check": "weights are safetensors", "pass": any(n.endswith(require_format) for n in manifest["files"])})
    checks.append({"check": "no pickle artefacts", "pass": not any(n.endswith(".pkl") for n in manifest["files"])})
    checks.append({"check": "model card present", "pass": "model-card.md" in manifest["files"]})
    result = pd.DataFrame(checks)
    return result, bool(result["pass"].all())


gate, ok = audit(workdir, manifest)
print("bundle promotable:", ok)
gate

bundle promotable: True


,check,pass
0,all file hashes match,True
1,weights are safetensors,True
2,no pickle artefacts,True
3,model card present,True


**Expected output** — `bundle promotable: True` and four passing checks, because the manifest was built before
the unsafe pickle was added. Add that pickle to the manifest and the format check fails — which is precisely the
behaviour you want from a gate that runs on every build.

In [8]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("temporary bundle removed:", not workdir.exists())

temporary bundle removed: True


**Expected output** — `temporary bundle removed: True`. Cleanup matters when a notebook writes files.

## Exercises

1. **Add a signature.** Use `hashlib` plus a keyed HMAC over the manifest and verify it. What does the gate gain,
   and what does it still not prove?
2. **Break the gate.** Add the pickle to the manifest and re-run `audit`. Which check fails, and what would a
   reviewer see?
3. **Pin dependencies.** Extend the manifest with a `requirements.lock` digest and explain how that changes the
   blast radius of a compromised dependency.
4. **Design the quarantine path.** Write the four steps a bundle follows when `audit` fails, starting from the
   moment the upload finishes.

## Reflection questions

1. Why is "the hash matches" not the same claim as "the model is safe"?
2. If an attacker can modify the manifest too, what control remains?
3. Which provenance field is most often missing in practice, and what does its absence cost during an incident?

## References

1. NIST, *SP 800-218*, Secure Software Development Framework, 2022 — supply chain.
2. Hugging Face, *safetensors* format documentation.
3. OWASP, *Top 10 for LLM Applications*, 2025 — LLM05 (supply chain).
4. SLSA specification, https://slsa.dev.